<a href="https://colab.research.google.com/github/corrielynnyuill-debug/AIProject_Stocks-CLY/blob/main/AIProject_Stocks_Part3_CLY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount drive in Colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the datasets
# Set file path
DATA_PATH = "/content/drive/MyDrive/AIProject/clean_data/"

# Load to DF
df = pd.read_parquet(DATA_PATH + "cleaned_stock_data.parquet")
print(df.head())
print('-'*80)
print(df.columns)


Mounted at /content/drive
  ticker      open     close  adj_close       low      high    volume  \
0      A -0.015269 -0.016592  -0.003238 -0.016609 -0.014101  4.657089   
1      A -0.015912 -0.017590  -0.003238 -0.016657 -0.015771  1.063811   
2      A -0.016320 -0.016592  -0.003238 -0.016592 -0.015532  0.406381   
3      A -0.016022 -0.017693  -0.003238 -0.016609 -0.015622  0.360644   
4      A -0.016618 -0.017401  -0.003238 -0.016609 -0.016024  0.274641   

        date   z_close   z_volume  close_outlier  volume_outlier  rolling_7  \
0 1999-11-18  0.053156  23.254405          False            True        NaN   
1 1999-11-19 -0.120629   4.522274          False            True        NaN   
2 1999-11-22  0.053156   1.095020          False           False        NaN   
3 1999-11-23 -0.138606   0.856593          False           False        NaN   
4 1999-11-24 -0.087669   0.408247          False           False        NaN   

   rolling_30  volatility  RSI         sector  \
0         N

In [2]:
# Feature engineering and technical indicators
# Apply EMA (Exponential Moving Average)
def ema(series, span):
  return series.ewm(span=span, adjust=False).mean()

# MACD (Moving Average Convergence Divergence) implementation
df = df.sort_values(['ticker', 'date'])

# MACD components
df['ema12'] = df.groupby('ticker')['close'].transform(lambda x: ema(x, span=12))
df['ema26'] = df.groupby('ticker')['close'].transform(lambda x: ema(x, span=26))

df['macd'] = df['ema12'] - df['ema26']
df['macd_signal'] = df.groupby('ticker')['macd'].transform(lambda x: ema(x, span=9))
df['macd_hist'] = df['macd'] - df['macd_signal']



print('-'*80)

--------------------------------------------------------------------------------


In [3]:
# MACD crossovers
df['macd_prev'] = df.groupby('ticker')['macd'].shift(1)
df['macd_signal_prev'] = df.groupby('ticker')['macd_signal'].shift(1)

# Buy when MACD crosses above signal
df['macd_buy'] = (
    (df['macd_prev'] < df['macd_signal_prev']) &
    (df['macd'] > df['macd_signal'])
)

# Sell when MACD crosses below signal
df['macd_sell'] = (
    (df['macd_prev'] > df['macd_signal_prev']) &
    (df['macd'] < df['macd_signal'])
)

print('-'*80)

--------------------------------------------------------------------------------


In [4]:
# RSI (Relative Strength Index) implementation with EMA
window = 14

# Price changes per ticker
delta = df.groupby('ticker')['close'].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

# EMA of gains and losses
avg_gain = gain.groupby(df['ticker']).transform(
    lambda x: x.ewm(alpha=1/window, adjust=False).mean())
avg_loss = loss.groupby(df['ticker']).transform(
    lambda x: x.ewm(alpha=1/window, adjust=False).mean())

RS = avg_gain / avg_loss
df['RSI'] = 100 - (100 / (1 + RS))

# RSI buy/sell signals
df['RSI_buy'] = df['RSI'] < 30
df['RSI_sell'] = df['RSI'] > 70

print('-'*80)



--------------------------------------------------------------------------------


In [ ]:
# Combine MACD and RSI
df['signal'] = 0 # default hold

df.loc[df['macd_buy'], df['RSI_buy'], 'signal'] = 1 # Buy
df.loc[df['macd_sell'], df['RSI_sell'], 'signal'] = -1 # Sell

print(df['signal'].value_counts())